In [1]:
import json
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage, ToolMessage

from life_insurance_business_analysis_assistant.poc.csv_query import execute_query

load_dotenv()

SYSTEM_PROMPT = """你是寿险业务报告的查询步骤执行 Agent。每次调用只处理输入中的一个模板步骤，不负责读取完整模板，也不负责组装或保存整份报告。

必须严格遵守以下规则：
1. 输入中的 current_step 是本次唯一要执行的步骤，不得增加、跳转或重复处理其他步骤。
2. 必须调用一次 execute_query，并将 current_step.data_source 原样作为 data_source 参数传入；查询、筛选、排序和字段投影全部以工具返回为准。
3. 只能根据 execute_query 返回的查询结果生成一条业务文案，不得猜测或增加工具结果中不存在的数字、机构和结论。
4. 最终只输出当前步骤的一条业务文案，不要输出标题、列表符号、JSON、执行过程或额外说明。
"""

model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    model_provider="deepseek",
)

agent = create_agent(
    model=model,
    tools=[execute_query],
    system_prompt=SYSTEM_PROMPT,
)



In [2]:
from pathlib import Path
from life_insurance_business_analysis_assistant.poc.template_loader import load_template

template_file = Path("./config/templates/月度分析报告模板.yaml")
template = load_template(template_file)



In [ ]:

for section in template.sections:
    section_name = section.name
    for step in section.steps:
        human_msg = HumanMessage(json.dumps(
                            {"current_step": step.model_dump(mode="json")},
                            ensure_ascii=False,
                        ))
        messages = [human_msg]
        response = agent.invoke(
            {"messages": messages}
        )
        print(response["messages"][-1].content)


全系统标保达成12444，标保达成率61.8%，标保同比下降11.6%；全年标保达成100538，全年标保进度53.8%，全年标保同比增长20.6%。
标保达成率高于70%的机构共有9家，按达成率从高到低依次为：新疆、天津、潍坊、云南、海南、河南、内蒙古、宁波、甘肃。
标保达成率低于50%的机构共有8家，依次为：贵州、安徽、山西、上海、青岛、宁夏、北京、深圳。
全年标保进度高于60%的机构共有7家，按进度从高到低依次为宁波、新疆、贵州、潍坊、天津、甘肃和云南。
全年标保进度低于45%的机构包括：安徽、海南、宁夏、青岛、北京、深圳。
全系统价值达成7045，价值达成率74.1%，价值同比提升9.8%，全年价值进度54.2%，全年价值同比提升7.2%。
价值达成率高于85%的机构为河南、天津、云南、吉林和新疆。
价值达成率低于55%的机构共有8家，分别为：宁波、贵州、山西、青岛、宁夏、北京、上海、深圳。


In [3]:
step = template.sections[0].steps[0]
step

Step(id=1, text='查询全系统的标保达成、标保达成率、标保同比、全年标保达成、全年标保进度、全年标保同比', data_source={'select': ['标保达成', '标保达成率', '标保同比', '全年标保达成', '全年标保进度', '全年标保同比'], 'from': 'data/标保.csv', 'where': [{'field': '片区', 'operator': 'eq', 'value': '全系统'}]})

In [11]:
stream = agent.stream_events(
    {
        "messages":[
            {
                "role":"user",
                "content":json.dumps(
                            {"current_step": step.model_dump(mode="json")},
                            ensure_ascii=False,
                        )
            }
        ]
    },
    version="v3"
)

In [4]:


stream = agent.stream_events(
    {
        "messages":[
            {
                "role":"user",
                "content":json.dumps(
                            {"current_step": step.model_dump(mode="json")},
                            ensure_ascii=False,
                        )
            }
        ]
    },
    version="v3"
)

for snapshot in stream.values:
    latest_message = snapshot["messages"][-1]

    if isinstance(latest_message,HumanMessage):
        print("HumanMessage:", latest_message)
    elif isinstance(latest_message, AIMessage):
        if latest_message.tool_calls:
            print("ToolCalls:", latest_message)
        else:
            print("AIMessage:", latest_message)
    elif isinstance(latest_message,ToolMessage):
        print("ToolMessage:", latest_message)

d:\Code\Data Agent\Life-Insurance-Business-Analysis-Assistant\.venv\Lib\site-packages\langgraph\pregel\main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
d:\Code\Data Agent\Life-Insurance-Business-Analysis-Assistant\.venv\Lib\site-packages\langgraph\pregel\main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


HumanMessage: content='{"current_step": {"id": 3, "text": "查询价值达成率低于55%的机构", "data_source": {"select": ["机构"], "from": "data/价值.csv", "where": [{"field": "价值达成率", "operator": "lt", "value": 55}], "order_by": [{"field": "价值达成率", "direction": "desc"}]}}}' additional_kwargs={} response_metadata={} id='4e1aa153-1c97-444f-9f67-5d609127f733'
ToolCalls: content=[{'type': 'reasoning', 'reasoning': 'Let me execute the query as instructed.', 'index': 0}, {'type': 'tool_call', 'id': 'call_00_erGClU9yye24ye12yUfT5499', 'name': 'execute_query', 'args': {'data_source': {'select': ['机构'], 'from': 'data/价值.csv', 'where': [{'field': '价值达成率', 'operator': 'lt', 'value': 55}], 'order_by': [{'field': '价值达成率', 'direction': 'desc'}]}}}] additional_kwargs={'reasoning_content': 'Let me execute the query as instructed.'} response_metadata={'model_provider': 'deepseek', 'finish_reason': 'tool_calls', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'output_version': 'v

In [ ]:
import csv
import re
from collections.abc import Mapping
from pathlib import Path
from typing import Annotated, Any

def _read_csv(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file, skipinitialspace=True)
        return [
            {(key or "").strip(): _parse_scalar(value) for key, value in row.items() if key is not None}
            for row in reader
        ]


def _parse_scalar(value: str | None) -> Any:
    if value is None:
        return None
    value = value.strip()
    if not value:
        return None
    numeric = value.replace(",", "")
    if re.fullmatch(r"[-+]?\d+", numeric):
        return int(numeric)
    if re.fullmatch(r"[-+]?(?:\d+\.\d*|\.\d+)", numeric):
        return float(numeric)
    return value


def _matches(row: Mapping[str, Any], condition: Mapping[str, Any]) -> bool:
    actual = row.get(condition["field"])
    expected = condition.get("value")
    if isinstance(actual, (int, float)) and isinstance(expected, str):
        expected = _parse_scalar(expected)

    operator = condition["operator"]
    if operator == "eq":
        return actual == expected
    if actual is None:
        return False
    if operator == "gt":
        return actual > expected
    if operator == "lt":
        return actual < expected
    if operator == "gte":
        return actual >= expected
    if operator == "lte":
        return actual <= expected
    raise ValueError(f"暂不支持 operator: {operator}")


def _sort_key(value: Any) -> tuple[bool, Any]:
    return (value is not None, value if value is not None else "")


def derive_facts(facts: list[dict[str, Any]]) -> dict[str, Any]:
    """Add the few deterministic values used by the text analyst."""

    directions: dict[str, str] = {}
    for row in facts:
        for field, value in row.items():
            if "同比" in str(field) and isinstance(value, (int, float)):
                directions[field] = "正增" if value > 0 else "负增" if value < 0 else "持平"
    count = len(facts)
    return {
        "count": count,
        "institution_count": count if any("机构" in row for row in facts) else None,
        "growth_direction": directions,
    }




In [ ]:
from langchain.tools import tool

@tool
def execute_query(data_source:dict[str,Any])->list[dict[str,Any]]:
    """按照模板步骤中的 data_source 配置查询 CSV 数据并返回结构化结果。"""
    source = Path(data_source["from"])
    print(source)
    rows = _read_csv(source)
    for condition in data_source.get("where", []) or []:
        rows = [row for row in rows if _matches(row, condition)]

    # Stable sorting in reverse order implements multi-column ordering.
    for ordering in reversed(data_source.get("order_by", []) or []):
        field = ordering["field"]
        reverse = ordering.get("direction", "asc").lower() == "desc"
        rows.sort(key=lambda row, name=field: _sort_key(row.get(name)), reverse=reverse)

    selected = data_source.get("select")
    if selected:
        rows = [{field: row.get(field) for field in selected} for row in rows]
    return rows



In [6]:
execute_query(step.data_source)

data\标保.csv


[{'标保达成': 12444,
  '标保达成率': 61.8,
  '标保同比': -11.6,
  '全年标保达成': 100538,
  '全年标保进度': 53.8,
  '全年标保同比': 20.6}]